In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import initialize_agent, AgentType
from langchain.tools import Tool
from langchain_community.tools import DuckDuckGoSearchRun

GOOGLE_API_KEY = "key"

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
    google_api_key=GOOGLE_API_KEY
)

llm_c = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.9,
    google_api_key=GOOGLE_API_KEY
)

llm_p = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
    google_api_key=GOOGLE_API_KEY
)

search = DuckDuckGoSearchRun()

In [50]:
tools = [search]

research_agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
)

# agent.run("cricket")
# topic = input("Enter a topic to research: ")

# — Planning-specific tools

def break_into_steps(task: str) -> str:
    """Breaks a goal into ordered steps."""
    return f"Breaking down: {task} into sequential steps for planning."

def estimate_timeline(task: str) -> str:
    """Estimates time required for a given task or plan."""
    return f"Estimating timeline for: {task}"

planning_tools = [
    Tool(
        name="TaskBreaker",
        func=break_into_steps,
        description=(
            "Use this to decompose a complex goal into smaller, "
            "manageable steps. Input should be a goal or objective."
        ),
    ),
    Tool(
        name="TimelineEstimator",
        func=estimate_timeline,
        description=(
            "Use this to estimate how long a task or full plan will take."
            "Input should be as task description."
        ),
    ),
]
# Planning Agent

planning_agent = initialize_agent(
    planning_tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose=True,
)

In [51]:
# writing-special-tools

def draft_content(topic: str) -> str:
    """Drafts an initial version of content on a given topic."""
    return f"Drafting initial content for: {topic}"

def generate_title(topic: str) -> str:
    """Generates compelling titles or headlines for the content."""
    return f"Generating title options for: {topic}"

writing_tools = [
    search,
    Tool(
        name="DraftWriter",
        func=draft_content,
        description="""
        Use this to create a first draft on web topics.
        Input should be the topic or title of the content.
        """
    ),
    Tool(
        name="TitleGenerator",
        func=generate_title,
        description="""
        Use this to generate main catchy titles or headlines.
        Input should be one topic or summary of the content.
        """
    ),
]

# — Writing Agent
writing_agent = initialize_agent(
    writing_tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose=True,
)

In [52]:
# — Editing-specific tools
def fix_grammar(text: str) -> str:
    """Fixes grammar, punctuation, and spelling errors."""
    return f"Fixing grammar and punctuation in: '{text[:100]}'..."

def plagiarism_check(text: str) -> str:
    """Flags potentially plagiarized or unoriginal sections."""
    return f"Scanning for plagiarism in: {text[:100]}..."

editing_tools = [
    search,

    Tool(
        name="GrammarFixer",
        func=fix_grammar,
        description=(
            "Use this to correct grammar, punctuation, and spelling. "
            "Input should be the text to fix."
        ),
    ),

    Tool(
        name="PlagiarismChecker",
        func=plagiarism_check,
        description=(
            "Use this to detect potentially plagiarized content. "
            "Input should be the text to scan."
        ),
    ),
]
# Editing Agent
editing_agent = initialize_agent(
    editing_tools,
    llm,
    Agent= AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose=True,
)

# Run
#article_to_edit = input("Paste your article/content to edit: ")

In [53]:
# Outline specific tools

def extract_key_sections(report: str) -> str:
    """Extracts all sections and headings from the report."""
    return f"Extracting key sections from report: {report[:100]}..."

def map_to_slides(sections: str) -> str:
    """Maps each report section to a corresponding slide."""
    return f"Mapping sections to slides: {sections[:100]}..."

def condense_to_bullets(section: str) -> str:
    """Condenses each section into 3-5 slide-ready bullet points."""
    return f"Condensing to bullet points: {section[:100]}..."

def write_slide_title(section: str) -> str:
    """Writes a smart, punchy title for each slide."""
    return f"Writing slide title for: {section[:100]}..."

def write_transition_notes(slide: str) -> str:
    """Adds transition phrases between slides for smooth flow."""
    return f"Writing transition notes for: {slide[:100]}..."

def identify_data_slides(report: str) -> str:
    """Identifies sections that should become charts or data visuals."""
    return f"Identifying data/chart opportunities in: {report[:50]}..."

def create_title_slide(topic: str) -> str:
    """Creates the opening title slide content."""
    return f"Creating title slide for: {topic}"

def create_summary_slide(report: str) -> str:
    """Creates a final summary/key takeaway slide."""
    return f"Creating summary slide from: {report[:100]}..."

def number_slides(outline: str) -> str:
    """Numbers and sequences all slides in correct order."""
    return f"Numbering and sequencing slides: {outline[:100]}..."

def estimate_slide_count(report: str) -> str:
    """Estimates how many slides the report will need."""
    return f"Estimating slide count for report: {report[:100]}..."
outline_tools = [
    search,
    Tool(
        name="SlideMapper",
        func=map_to_slides,
        description="""
        Use this to map each report section to a slide.
        Input should be the list of extracted sections.
        """,
    ),
    Tool(
        name="SlideTitleWriter",
        func=write_slide_title,
        description="""
        Use this to write a short punchy title for each slide.
        Input should be the section heading or topic of the slide.
        """,
    ),
    Tool(
        name="SlideCountEstimator",
        func=estimate_slide_count,
        description="""
        Use this to estimate the total number of slides needed.
        Input should be the full report text.
        """,
    ),
]

# — Presentation Outline Agent —

presentation_outline_agent = initialize_agent(
    outline_tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose=True,
)

# — Run —
# final_report = input("Paste your final edited report: ")


In [54]:
# FULL 5-AGENT PIPELINE

# = Single user Input

goal = input("Enter your goal: ")

# AGENT 1 - RESEARCH AGENT
# Input : user goal
# Output : research_output

prompt = (
    f"Research the topic: {goal}. "
    "Use DuckDuckGo for recent developments. "
    "Gather key facts, statistics. "
    "Compile everything into a detailed summary."
)

#research_output = research_agent.run(prompt)

#planning_agent.run(planning_prompt)

research_output = research_agent.run(prompt)

print("** AGENT 1 - Research Done\n")

time.sleep(10)

# AGENT 2 - PLANNING AGENT
# Input : user goal
# Output : planning_output

planning_prompt = f"""
Goal: {goal}

Research Summary:
{research_output}

Using the goal and research above,
- Create a detailed, actionable plan.
- Break the goal into clear steps using TaskBreaker.
- Estimate the total timeline using TimelineEstimator.
- Output a structured plan with phases and deadlines.
"""

#planning_agent.run(planning_prompt)

planning_output = planning_agent.run(planning_prompt)
print("** AGENT 2 - Planning Done\n")
time.sleep(10)
# AGENT 3 - WRITING AGENT
# Input : research_output + planning_output
# Output : writing_output

writing_prompt = (
    f"Write a detailed, engaging article about: {planning_output}. "
    "Use DuckDuckGo to gather the latest facts and references. "
    "Use DraftWriter to create the initial draft. "
    "Output the final polished article with title, introduction, "
    "body sections, and a conclusion."
)

#writing_agent.run(writing_prompt)

writing_output = writing_agent.run(writing_prompt)
print("** AGENT 3 - Writing Done\n")
time.sleep(10)
# AGENT 4 - EDITING AGENT
# Input = writing_output
# Output = editing_output

editing_prompt = f"""
Edit and reword the following content:\n\n{writing_output}\n\n
Use @GrammarFixer to correct any language errors.
Use PlagiarismChecker to flag any unoriginal content.
Output the fully edited, polished final version.
"""

#editing_agent.run(editing_prompt)

editing_output = editing_agent.run(editing_prompt)
print("** AGENT 4 - Editing Done\n")
time.sleep(10)
# AGENT 5 - PRESENTATION OUTLINE AGENT
# Input = editing_output
# Output = final_presentation

outline_prompt = f"""
Convert the following report into a presentation-ready outline:

{editing_output}

Follow these steps:
1. Use SlideCountIdentifier to determine how many slides are needed.
2. Use SlideMapper to assign each section to a slide.
3. Use SlideTitleIdentifier to write a punchy title for every slide.

Output a clean, numbered, slide-by-slide outline in EXACTLY this format for each slide:

SLIDE [N] – [TITLE]

- [Bullet Point 1]: [1-2 sentence description expanding on this point with key details, context, or supporting data.]
- [Bullet Point 2]: [1-2 sentence description expanding on this point with key details, context, or supporting data.]
- [Bullet Point 3]: [1-2 sentence description expanding on this point with key details, context, or supporting data.]

Rules:
- Every bullet must have a short bold label (the point) followed by a colon and a 1-2 sentence description.
- Descriptions should add context, stats, or explanation — not just repeat the bullet label.
- Keep bullet labels concise (3-6 words max).
- Descriptions should be copy-paste ready for PowerPoint speaker notes or slide body text.
- Do not add any extra commentary outside the slide format above.
"""

#presentation_outline_age   nt.run(outline_prompt)

final_presentation = presentation_outline_agent.run(outline_prompt)
print("** AGENT 5 – Presentation Done\n")
time.sleep(10)
print("-" * 60)
print(" FINAL PRESENTATION OUTLINE")
print("-" * 60)
print(final_presentation)




> Entering new AgentExecutor chain...
Thought: To research the topic of "HOW TO MAKE COLD COFFEE," I should start by looking for general methods and recipes online.

Action: duckduckgo_search
Action Input: "how to make cold coffee"
Observation: This refreshing creamy instant iced coffee drink will give your Starbucks coffee a run for its money, this recipe is extremely easy, quick and super affordab... With a tiny bit of prep work, you can have smooth cold brew coffee in your life week after delicious week. We’ll show you how using three different technique... Learn how to make cold brew coffee at home, from picking the right type of coffee to the easy brewing method.
Thought:Question: Research the topic: HOW TO MAKE COLD COFFEE. Use DuckDuckGo for recent developments. Gather key facts, statistics. Compile everything into a detailed summary.
Thought: To research the topic of "HOW TO MAKE COLD COFFEE," I should start by looking for general methods and recipes online.

Action: duckduck

TimeoutException: Request timed out: RuntimeError('error sending request for url (https://search.brave.com/search?q=how+to+make+cold+coffee&source=web&tf=pm): operation timed out\n\nCaused by:\n    operation timed out')